## Sel 1: Import Pustaka

In [1]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb

from sklearn.multioutput import MultiOutputRegressor

import joblib
import contextlib
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

## Sel 2: Memuat & Pra-pemrosesan Data (Resampling Harian)

In [2]:
# 1. BACA DATASET (Ganti dengan nama file aslimu)
df = pd.read_csv('dataset_polutan_jatim60.csv')

# --- BAGIAN INI HARUS DISESUAIKAN DENGAN NAMA KOLOM ASLI DI EXCEL-MU ---
kolom_waktu = 'Waktu'       # Kolom yang berisi tanggal dan jam
kolom_kota = 'Kota'         # Kolom nama kota/kabupaten
polutan = ['PM2.5 (µg/m³)', 'PM10 (µg/m³)','SO2 (µg/m³)', 'CO (µg/m³)', 'NO2 (µg/m³)', 'Ozon (µg/m³)'] # Pastikan namanya persis sama
# -----------------------------------------------------------------------

# 2. Ubah format teks menjadi tipe Datetime yang dikenali Python
df[kolom_waktu] = pd.to_datetime(df[kolom_waktu])

# 3. Jadikan Waktu dan Kota sebagai Index agar mudah di-resample
df.set_index(kolom_waktu, inplace=True)

# 4. Resample: Hitung rata-rata polutan per HARI (huruf 'D' = Daily) untuk setiap Kota
# Cara ini akan memadatkan 24 baris (per jam) menjadi 1 baris per hari
df_daily = df.groupby(kolom_kota)[polutan].resample('D').mean().reset_index()

# Hapus baris yang kosong (misal ada hari dimana sensor mati total)
df_daily = df_daily.dropna()

print(f"Bentuk data setelah dijadikan rata-rata harian: {df_daily.shape}")
df_daily.head()

Bentuk data setelah dijadikan rata-rata harian: (2318, 8)


,Kota,Waktu,PM2.5 (µg/m³),PM10 (µg/m³),SO2 (µg/m³),CO (µg/m³),NO2 (µg/m³),Ozon (µg/m³)
0,Kabupaten Bangkalan,2026-02-22,17.432143,20.432143,1.152143,548.186429,5.871429,40.233571
1,Kabupaten Bangkalan,2026-02-23,13.778750,16.843750,0.712500,427.443750,4.183750,20.073750
2,Kabupaten Bangkalan,2026-02-24,7.332500,9.698125,1.066250,337.213125,3.926875,34.053125
3,Kabupaten Bangkalan,2026-02-25,7.234167,10.883333,1.179167,354.936250,4.758333,21.076667
4,Kabupaten Bangkalan,2026-02-26,2.575417,4.392500,0.875000,180.410833,2.883333,20.656250


## Sel 3: Rekayasa Fitur (Sliding Window 3 Hari & Target Besok)

In [3]:
# Fungsi untuk membuat Sliding Window (t-1, t-2, t-3) dan Target (t+1)
def create_sliding_window(df_kota, n_lags=3):
    df_temp = df_kota.copy()
    
    # Buat fitur history (t-1, t-2, t-3) untuk SETIAP polutan
    for p in polutan:
        for i in range(1, n_lags + 1):
            df_temp[f'{p}_H-{i}'] = df_temp[p].shift(i)
            
    # Buat target yang akan ditebak (t+1 alias BESOK)
    for p in polutan:
        df_temp[f'TARGET_{p}_Besok'] = df_temp[p].shift(-1)
        
    return df_temp

# Aplikasikan fungsi di atas untuk masing-masing kota secara terpisah agar tidak bentrok
df_model = df_daily.groupby(kolom_kota, group_keys=False).apply(create_sliding_window)

# Hapus baris yang memiliki NaN (akibat proses pergeseran hari/shift)
df_model = df_model.dropna().reset_index(drop=True)

# Lakukan One-Hot Encoding untuk kolom Kota (Mengubah teks kota menjadi kolom 0 dan 1)
df_model = pd.get_dummies(df_model, columns=[kolom_kota])

# ======================================================================
# BAGIAN YANG HILANG: MEMECAH DATA MENJADI X (Fitur) DAN y (Target)
# ======================================================================

# 1. Kumpulkan semua nama kolom yang merupakan jawaban (Target Besok)
kolom_target = [col for col in df_model.columns if 'TARGET_' in col]

# 2. Pisahkan tabel jawaban menjadi y
y = df_model[kolom_target]

# 3. Pisahkan tabel soal menjadi X 
# (Membuang kolom jawaban dan kolom 'Waktu' karena AI tidak bisa membaca tanggal)
X = df_model.drop(columns=kolom_target + [kolom_waktu])

print(f"Bentuk tabel awal df_model : {df_model.shape}")
print(f"Dimensi X (Fitur AI)       : {X.shape}")
print(f"Dimensi y (Target AI)      : {y.shape}")

Bentuk tabel awal df_model : (2166, 69)
Dimensi X (Fitur AI)       : (2166, 62)
Dimensi y (Target AI)      : (2166, 6)


## Sel 4: Splitting & Training Baseline Model MLFLOW

In [4]:
# 1. Splitting Data (80% Training, 20% Testing) 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Set nama eksperimen utama di MLflow
mlflow.set_experiment("Proyek_ISPU_Jatim")

# 2. Persiapan 3 Baseline Model
baselines = {
    "Baseline_LinearRegression": MultiOutputRegressor(LinearRegression()),
    "Baseline_RandomForest": MultiOutputRegressor(RandomForestRegressor(n_estimators=50, random_state=42)),
    "Baseline_KNeighbors": MultiOutputRegressor(KNeighborsRegressor())
}

print("--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---")
for nama_model, model in baselines.items():
    # Membuka sesi pencatatan MLflow untuk setiap model
    with mlflow.start_run(run_name=nama_model):
        
        # Training & Prediksi
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Kalkulasi Metrik Keseluruhan
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        # Mencatat parameter, metrik, dan artifact model ke MLflow
        mlflow.log_param("algoritma", nama_model)
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_r2", r2)
        mlflow.sklearn.log_model(model, artifact_path=nama_model)
        
        print(f"✅ {nama_model} berhasil dicatat (RMSE: {rmse:.2f})")

2026/05/05 08:02:04 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/05 08:02:04 INFO mlflow.store.db.utils: Updating database tables
2026/05/05 08:02:06 INFO mlflow.tracking.fluent: Experiment with name 'Proyek_ISPU_Jatim' does not exist. Creating a new experiment.


--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---


2026/05/05 08:02:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 08:02:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline_LinearRegression berhasil dicatat (RMSE: 39.19)


2026/05/05 08:03:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 08:03:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline_RandomForest berhasil dicatat (RMSE: 28.69)


  File "c:\Users\Tigro\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\Tigro\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Tigro\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\Tigro\AppData\Local\Programs\Python\Python311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
2026/05/05 08:03:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/05 08:03:28 WARNING mlflow.sklearn: Saving scikit-learn models in the

✅ Baseline_KNeighbors berhasil dicatat (RMSE: 39.19)


## Sel 5 : Hyperparameter Tuning XGBoost MLFlow

In [5]:
print("Memulai Hyperparameter Tuning XGBoost...")

# 1. Inisiasi Model Dasar
model_xgb_dasar = MultiOutputRegressor(xgb.XGBRegressor(random_state=42))

# 2. Menentukan Ruang Pencarian Ruang (Grid)
param_grid = {
    'estimator__n_estimators': [100, 200, 300, 400],
    'estimator__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'estimator__max_depth': [3, 5, 7, 9],
    'estimator__subsample': [0.8, 0.9, 1.0],
    'estimator__colsample_bytree': [0.8, 0.9, 1.0]
}

# 3. Mempersiapkan Mesin Pencari (Randomized Search)
grid_search = RandomizedSearchCV(
    estimator=model_xgb_dasar, 
    param_distributions=param_grid, 
    n_iter=30, 
    cv=4,      
    scoring='neg_mean_absolute_error', 
    random_state=42,
    n_jobs=-1, 
    verbose=0  
)

# --- JEMBATAN TQDM & JOBLIB ---
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
# ------------------------------

# === MEMULAI TRACKING TUNING DI MLFLOW ===
with mlflow.start_run(run_name="Tuning_XGBoost_RandomSearch"):
    
    # Mencatat batasan iterasi ke MLflow
    mlflow.log_param("n_iter_search", 30)
    mlflow.log_param("cv_folds", 4)
    
    # 4. Eksekusi Pencarian dengan Progress Bar
    total_fits = grid_search.n_iter * grid_search.cv 
    with tqdm_joblib(tqdm(desc="Progres Tuning XGBoost", total=total_fits, unit="fit")):
        grid_search.fit(X_train, y_train)

    # 5. Mengambil Otak Terbaik
    model_terbaik = grid_search.best_estimator_

    print("\n--- TUNING SELESAI ---")
    print(f"Kombinasi Parameter Terbaik:\n{grid_search.best_params_}")
    
    # Log setiap hyperparameter terbaik ke MLflow
    for param_name, param_val in grid_search.best_params_.items():
        mlflow.log_param(param_name, param_val)

    # Melakukan prediksi ke data ujian
    y_pred = model_terbaik.predict(X_test)

    # Daftar nama target
    polutan_labels = ['PM25', 'PM10','SO2', 'CO', 'NO2', 'O3'] 

    print("--- EVALUASI MODEL TERBAIK PER POLUTAN ---")
    
    total_mae, total_rmse = 0, 0
    
    for i, polutan in enumerate(polutan_labels):
        mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred[:, i]))
        rata_rata_asli = y_test.iloc[:, i].mean()
        
        total_mae += mae
        total_rmse += rmse
        
        if rata_rata_asli > 0:
            persentase_error = (mae / rata_rata_asli) * 100
        else:
            persentase_error = 0.0
            
        print(f"\nPolutan {polutan}:")
        print(f"  - Rata-rata asli : {rata_rata_asli:.2f}")
        print(f"  - MAE (Meleset)  : {mae:.2f}")
        print(f"  - RMSE           : {rmse:.2f}")
        print(f"  - Error (%)      : {persentase_error:.2f}%")
        
        # Log metrik per polutan ke MLflow
        mlflow.log_metric(f"mae_{polutan}", mae)
        mlflow.log_metric(f"rmse_{polutan}", rmse)

    # Log rata-rata performa keseluruhan ke MLflow agar bisa dibandingkan dengan baseline
    mlflow.log_metric("test_mae", total_mae / len(polutan_labels))
    mlflow.log_metric("test_rmse", total_rmse / len(polutan_labels))
    
    # Memenuhi syarat "explore penggunaan MLflow Model"
    mlflow.sklearn.log_model(model_terbaik, artifact_path="Best_XGBoost_Model")

Memulai Hyperparameter Tuning XGBoost...


Progres Tuning XGBoost:   0%|          | 0/120 [00:00<?, ?fit/s]


--- TUNING SELESAI ---
Kombinasi Parameter Terbaik:
{'estimator__subsample': 0.9, 'estimator__n_estimators': 400, 'estimator__max_depth': 7, 'estimator__learning_rate': 0.05, 'estimator__colsample_bytree': 0.8}
--- EVALUASI MODEL TERBAIK PER POLUTAN ---

Polutan PM25:
  - Rata-rata asli : 7.99
  - MAE (Meleset)  : 1.86
  - RMSE           : 3.66
  - Error (%)      : 23.33%

Polutan PM10:
  - Rata-rata asli : 11.52
  - MAE (Meleset)  : 2.30
  - RMSE           : 4.13
  - Error (%)      : 19.93%

Polutan SO2:
  - Rata-rata asli : 0.59
  - MAE (Meleset)  : 0.13
  - RMSE           : 0.30
  - Error (%)      : 22.75%

Polutan CO:
  - Rata-rata asli : 205.68
  - MAE (Meleset)  : 32.93
  - RMSE           : 62.84
  - Error (%)      : 16.01%


2026/05/05 08:07:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Polutan NO2:
  - Rata-rata asli : 1.83
  - MAE (Meleset)  : 0.49
  - RMSE           : 1.16
  - Error (%)      : 26.55%

Polutan O3:
  - Rata-rata asli : 33.99
  - MAE (Meleset)  : 2.88
  - RMSE           : 4.80
  - Error (%)      : 8.49%


2026/05/05 08:07:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


## Sel 6: Ekspor Model (Menyimpan Hasil)

In [6]:
# Membungkus model dan nama kolom agar Backend tidak kebingungan
paket_model = {
    'model': model_terbaik,
    'fitur': list(X_train.columns)
}

# Menyimpan ke dalam file Pickle
nama_file = 'xgb_ispu_jatim.pkl'
joblib.dump(paket_model, nama_file)

print(f"Selesai! Model pintar telah berhasil diekspor sebagai '{nama_file}'")
print("Jangan lupa salin file ini dan timpa file lama yang ada di dalam folder 'backend/models/'!")

Selesai! Model pintar telah berhasil diekspor sebagai 'xgb_ispu_jatim.pkl'
Jangan lupa salin file ini dan timpa file lama yang ada di dalam folder 'backend/models/'!
